# 02 · QC, trimming and kallisto: the decisions and what the results show
**Goal:** check the reads, remove sequencing artefacts, and estimate how many reads come from each transcript.
**Input:** results of the heavy steps. **Output:** `results/kallisto/<SRR>/` (estimates), `results/qc/` (quality reports; the small fastp JSON ones are in git).
**Kernel:** Python 3 · **Time:** minutes to read the results.

**The heavy steps run on Euler, not on the laptop:** full-read FastQC, trimming and kallisto are submitted as cluster jobs (notebook 01, `euler/03_process_array.sh`); copy the results back and this notebook shows them and records the trimming decision. To run the steps on this laptop instead (hours, fan at full speed, ~70 GB disk) set `RUN_LOCALLY = True` below; every step skips finished work.

Per run, four steps (FASTQ files are 7–10 GB each and deleted after use; the `.sra` stays):

| Step | Tool | What it does |
|---|---|---|
| 1 Convert | `fasterq-dump` | `.sra` → **FASTQ**: text files of reads with a quality score per base, two per run (mate 1, mate 2) |
| 2 Check | **FastQC**, **MultiQC** | per-base quality, adapter content, duplication, per sample; MultiQC combines all samples in one report |
| 3 Clean | **fastp** | cuts **adapter** sequence and low-quality ends, drops low-quality or too-short reads |
| 4 Count | **kallisto** | finds which *transcripts* each read pair is compatible with and estimates reads per transcript (*pseudoalignment*: fast, no exact genomic position needed). Transcript → gene comes from the annotation and is applied later (notebook 03; [docs/pipeline.md](../docs/pipeline.md)) |

Terms (FASTQ, adapter, pseudoalignment, …): [docs/glossary.md](../docs/glossary.md).

In [1]:
# helpers.py: pipeline functions; runs = sample table
import sys; sys.path.insert(0, '../scripts')
import helpers as h
from IPython.display import display
runs = h.load_runs()
RUN_LOCALLY = False   # False (default): heavy steps run on Euler (notebook 01); True: run them on this laptop

## 1 · Reference and index (local only, ~2 min)
Reads are matched against the sequences of all annotated mouse transcripts (**GENCODE release vM25**, genome version GRCm38/mm10). kallisto first builds an **index**, a searchable table of that reference. On Euler this is `euler/02b_kallisto_index.sh`.

In [2]:
# local only: download the reference files (skipped if present) and build the kallisto index
if RUN_LOCALLY:
    h.download_reference()
    h.build_index()

## 2 · Quality control of all reads
Every run is checked on **all** reads (a few minutes per run; on Euler this is the `qc` stage of `euler/03_process_array.sh`). What to look at in the FastQC table:

The verdicts are FastQC's own (`summary.txt` in each report), from fixed thresholds in its `limits.txt`:

| Check | Meaning | WARN when | FAIL when | Expected for RNA-seq |
|---|---|---|---|---|
| Per base sequence quality | quality along the read | lower quartile < 10 or median < 25 at any position | lower quartile < 5 or median < 20 | PASS |
| Per sequence quality scores | quality per read | most common mean quality < 27 | < 20 | PASS |
| **Adapter Content** | share of reads running into adapter sequence | > 5 % of reads | > 10 % | PASS, else trim |
| Overrepresented sequences | identical sequences far more common than chance | one sequence > 0.1 % of reads | > 1 % | WARN is common (here: Smart-seq2 primers, each ≤ 0.25 % in the sample checked) |
| Sequence Duplication Levels | repeated reads | < 70 % of reads remain after removing duplicates | < 50 % | WARN is normal: highly expressed genes give identical reads |

**Result for these 27 runs:** base quality PASS in all 54 read files; **adapter content FAIL in all 54**: 17–34 % of reads run into Nextera transposase sequence at their ends (the Smart-seq2 library has fragments shorter than the 150-base read); overrepresented sequences and duplication WARN, as expected.

In [3]:
# FastQC on every downloaded run (local only), then one combined MultiQC report; the table shows the verdicts of whatever reports exist (from Euler or local)
if RUN_LOCALLY:
    for srr in [s for s in runs.index if h.has_sra(s)]:
        h.fastqc(srr)
    print('report:', h.multiqc())   # open this HTML: all samples side by side
h.fastqc_summary(list(runs.index))

,Per base sequence quality,Per sequence quality scores,Adapter Content,Overrepresented sequences,Sequence Duplication Levels
SRR21355401_1,PASS,PASS,FAIL,WARN,WARN
SRR21355401_2,PASS,PASS,FAIL,WARN,WARN
SRR21355396_1,PASS,PASS,FAIL,WARN,WARN
SRR21355396_2,PASS,PASS,FAIL,WARN,WARN
SRR21355392_1,PASS,PASS,FAIL,WARN,WARN
SRR21355392_2,PASS,PASS,FAIL,WARN,WARN
SRR21355388_1,PASS,PASS,FAIL,WARN,WARN
SRR21355388_2,PASS,PASS,FAIL,WARN,WARN
SRR21355398_1,PASS,PASS,FAIL,WARN,WARN
SRR21355398_2,PASS,PASS,FAIL,WARN,WARN


## 3 · Trim or not?
Adapter content fails in **every** sample, so **all** runs are trimmed with the same settings. Trimming some samples and not others would create a technical difference between groups. fastp detects the adapter itself (paired-end overlap), keeps reads ≥ 20 bases and also drops reads with many low-quality bases (about 4–5 %). Duplicates are deliberately **not** removed (glossary: *Duplicates*).

In [4]:
# decision recorded for all later cells; the guard in step 5 raises if any run was processed differently
TRIM = True   # FastQC on all 27 runs: Nextera adapter FAIL (17-34 % of reads) + poly-G in every sample; quality PASS

### Check: adapter and poly-G (SRR21355398, first 600,000 read pairs)
**Is it the adapter?** Yes.
- The 12 bases `CTGTCTCTTATA` start the *Nextera* adapter (expected: Tn5 library). They occur in 27–30 % of reads, vs ≈ 5 expected by chance.
- The bases after them match the read's *own* mate's adapter in 96–99 % of cases, the other mate's in 0 % (the mates' adapters differ after 19 bases).
- After trimming, FastQC adapter content drops from 25–34 % to 0.0 % (two runs checked).

fastp cuts from the adapter to the read end (~52 bases per read here, more than the 34-base adapter), so anything behind the adapter is removed too.

**Poly-G** (≥ 12 G in a row). *How:* the NovaSeq images each cycle in two colours: T = green, C = red, A = both, **G = neither**, so a cluster that emits no light is read as G. *Why here:* a read has 150 cycles. After a short insert it reads the adapter, a sample barcode and a further stretch of the library construct, then **runs out of template**: every remaining cycle is dark.

```
read    [ insert ][ adapter ][ barcode ][ rest of construct ][ no template left: dark -> GGGGGGG ]
bases      ~49         34          8            ~33                  ~24   (= 150 - (49 + 75))
```
(medians of the poly-G reads of SRR21355398.) The *barcode* is the 8 constant bases after the adapter: each of the 27 runs has its own (81–96 % of that run's adapter reads), no two alike, as a sample index would. A 12-G run needs an insert shorter than ~60 bases, so only 4–6 % of reads have one although 27–30 % reach the adapter. The G's get high quality (mean Q32): the score reflects signal clarity and a consistently dark cluster is clear, so quality trimming misses them. They lie behind the adapter (79–88 % of poly-G reads) and are cut with it: 91–93 % lose them, 5–6 % are dropped by fastp's filters, 1.5–3 % keep a G run. Of those, 96 % have no adapter in the read: about half are real G-rich sequence inside the read, the rest end in G's. fastp's `--trim_poly_g` catches these, so **it is switched on for all 27 runs** (the percentages above describe adapter trimming alone). Tested on one run before the switch: pseudoalignment is identical (65.20 %) and total counts change by 0.05 %, but a few G-rich genes lose 58–96 % of their counts (Gm19220, Gm26822, Grip2: kallisto assigns residual poly-G reads to them; `results/qc/polyG_sensitivity_SRR21355398.csv`). None is differentially expressed in any comparison, so conclusions were unaffected; with poly-G trimming on, those inflated counts are gone.

In [5]:
# per mate: adapter evidence and the fate of poly-G reads (needs the .sra on this machine; re-runs fastq-dump and fastp on 600k pairs, ~1 min)
if h.has_sra('SRR21355398'):
    display(h.adapter_check('SRR21355398').T)
else:
    print('SRR21355398.sra is not on this machine (notebook 00 downloads it); the results are described in the text above')

mate,1,2
reads,600000.0,600000.0
adapter_start_%,27.2,29.6
expected_by_chance_reads,5.0,5.0
continues_own_adapter_%,95.8,98.5
matches_other_mate_adapter_%,0.0,0.0
polyG_reads_%,6.2,4.3
polyG_after_adapter_%,78.8,88.0
polyG_dropped_%,5.2,5.9
polyG_removed_%,93.3,91.1
polyG_remaining_%,1.5,3.0


## 4 · Test on one run (local only)
The smallest run, SRR21355398, first, before committing to all 27.

In [6]:
# local only: convert -> trim -> kallisto for one run; then the summary table of all runs that have results
if RUN_LOCALLY:
    h.process('SRR21355398', trim_reads=TRIM)
h.qc_summary(runs)

,pairs_M,aligned_%,q30_%,dup_%,group
SRR21355401,14.9,63.0,89.7,7.6,aging 3mo
SRR21355396,25.7,64.6,87.4,6.9,aging 3mo
SRR21355392,16.1,64.8,88.2,6.6,aging 3mo
SRR21355388,16.9,65.0,88.5,7.2,aging 3mo
SRR21355398,6.4,65.2,86.5,3.3,aging 18mo
SRR21355394,7.8,65.3,86.2,2.9,aging 18mo
SRR21355390,28.2,61.9,88.7,8.0,aging 18mo
SRR21355386,9.3,65.5,86.1,3.3,aging 18mo
SRR21355397,14.2,64.1,88.5,5.6,aging 21mo
SRR21355393,9.8,67.6,87.7,3.8,aging 21mo


Expect a **pseudoaligned** share of 60–80 % of read pairs for mouse mRNA (here 62–71 %). A much lower value would point to a wrong reference or a problem with the library.

**Effect of trimming:** for the five runs that were first quantified without trimming, the pseudoaligned share rose by 2.1–2.7 points after trimming (e.g. 62.6 → 65.2 % for SRR21355398).

## 5 · All runs (local only)
Skips finished runs; re-run after an interruption. The guard before and after the loop raises an error if any run was processed with a different trimming setting (on Euler all runs get the same script).

In [7]:
# process every downloaded run (local only); the guard checks that all runs share the same trimming
print('TRIM =', TRIM)   # must be True: re-run the TRIM cell if the kernel was restarted
h.check_consistent(runs, TRIM)
if RUN_LOCALLY:
    for srr in runs.index:
        if h.has_sra(srr):
            h.process(srr, trim_reads=TRIM)
    h.clean_tmp()
    h.check_consistent(runs, TRIM)

TRIM = True
consistent: all quantified runs used trimmed=True


## 6 · Summary
`pairs_M` = millions of read pairs; `aligned_%` = share pseudoaligned by kallisto; `q30_%` = share of bases with error rate ≤ 1 in 1,000; `dup_%` = fastp's duplicate estimate (3–8 %, low). `trimmed` should be `True` for all runs.

In [8]:
# per-run QC numbers from kallisto and fastp
h.qc_summary(runs)

,pairs_M,aligned_%,q30_%,dup_%,group
SRR21355401,14.9,63.0,89.7,7.6,aging 3mo
SRR21355396,25.7,64.6,87.4,6.9,aging 3mo
SRR21355392,16.1,64.8,88.2,6.6,aging 3mo
SRR21355388,16.9,65.0,88.5,7.2,aging 3mo
SRR21355398,6.4,65.2,86.5,3.3,aging 18mo
SRR21355394,7.8,65.3,86.2,2.9,aging 18mo
SRR21355390,28.2,61.9,88.7,8.0,aging 18mo
SRR21355386,9.3,65.5,86.1,3.3,aging 18mo
SRR21355397,14.2,64.1,88.5,5.6,aging 21mo
SRR21355393,9.8,67.6,87.7,3.8,aging 21mo


In [9]:
# downloaded / quantified / trimmed per run
h.status(runs)

,group,sra_GB,quantified,trimmed
SRR,,,,
SRR21355401,aging 3mo,1.67,True,True
SRR21355396,aging 3mo,3.03,True,True
SRR21355392,aging 3mo,1.88,True,True
SRR21355388,aging 3mo,1.94,True,True
SRR21355398,aging 18mo,0.78,True,True
SRR21355394,aging 18mo,0.95,True,True
SRR21355390,aging 18mo,3.23,True,True
SRR21355386,aging 18mo,1.13,True,True
SRR21355397,aging 21mo,1.66,True,True
